In [1]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd

def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'fbn_pipeline').exists():
            return candidate
    raise RuntimeError('Project root not found')

PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fbn_pipeline import (
    ExperimentConfig,
    build_feature_matrix,
    load_training_split,
    lock_development_config,
    run_baseline_development,
    run_development,
    run_final_evaluation_from_lock,
    select_best_baseline,
    select_best_development_config,
    write_baseline_lock,
)
from fbn_pipeline.baselines import baseline_lock_payload
from fbn_pipeline.metrics import per_subject_metrics

SEED = 42
K_FOLDS = 5
BOOTSTRAP_SEED = 4242
BOOTSTRAP_REPLICATES = 10000
PRIMARY_SELECTION_METRIC = 'oof_balanced_accuracy'
TIE_BREAK_RULES = [
    'oof_macro_f1',
    'oof_auc',
    'lower_model_complexity',
    'configuration_id_lexicographic',
]
FINAL_REPORTING_METRICS = [
    'accuracy',
    'balanced_accuracy',
    'precision',
    'recall',
    'task_f1',
    'macro_f1',
    'roc_auc',
    'confusion_matrix',
    'per_subject_accuracy',
    'per_subject_balanced_accuracy',
    'per_subject_task_f1',
    'per_subject_roc_auc_when_defined',
]
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'publication_protocol_v2'
DEVELOPMENT_DIR = ARTIFACT_ROOT / 'development'
COMPARISON_DIR = ARTIFACT_ROOT / 'comparison'
BASELINE_DIR = ARTIFACT_ROOT / 'baselines'
BASELINE_LOCK_DIR = BASELINE_DIR / 'locks'
ABLATION_LOCK_DIR = ARTIFACT_ROOT / 'ablations' / 'locks'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_LOCK_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_LOCK_DIR.mkdir(parents=True, exist_ok=True)
PROPOSED_LOCK_PATH = ARTIFACT_ROOT / 'locked_config.json'
PROPOSED_NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'publication_protocol_v2_development.ipynb'
COMPARISON_MANIFEST_PATH = COMPARISON_DIR / 'final_comparison_manifest.json'
STAT_PLAN_JSON_PATH = COMPARISON_DIR / 'statistical_analysis_plan.json'
STAT_PLAN_MD_PATH = COMPARISON_DIR / 'statistical_analysis_plan.md'
BASELINE_CANDIDATES_CSV_PATH = BASELINE_DIR / 'baseline_candidates_oof.csv'
BASELINE_WINNERS_CSV_PATH = BASELINE_DIR / 'baseline_family_winners_oof.csv'
BASELINE_SUMMARY_JSON_PATH = BASELINE_DIR / 'baseline_development_summary.json'
BASELINE_SUMMARY_MD_PATH = BASELINE_DIR / 'baseline_development_summary.md'
random.seed(SEED)
np.random.seed(SEED)

def stable_hash(obj) -> str:
    import hashlib
    payload = json.dumps(obj, sort_keys=True, separators=(',', ':'), default=float)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def array_hash(values: np.ndarray) -> str:
    import hashlib
    arr = np.ascontiguousarray(values, dtype=np.float64)
    return hashlib.sha256(arr.tobytes()).hexdigest()

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def config_hash_from_dict(cfg_dict: dict) -> str:
    return stable_hash(cfg_dict)

def family_display_name(family_key: str) -> str:
    mapping = {
        'stat_logreg': 'Statistical features + Logistic Regression',
        'stat_svm': 'Statistical features + RBF SVM',
        'stat_random_forest': 'Statistical features + Random Forest',
        'stat_hist_gradient_boosting': 'Statistical features + HistGradientBoosting',
        'flat_logreg': 'Flattened raw window + Logistic Regression',
    }
    return mapping[family_key]

def baseline_grid() -> dict:
    return {
        'stat_logreg': {
            'representation': 'statistical_32',
            'candidates': [
                {'C': C, 'class_weight': class_weight, 'solver': 'lbfgs', 'max_iter': 5000}
                for C, class_weight in product([0.1, 1.0, 10.0], [None, 'balanced'])
            ],
        },
        'stat_svm': {
            'representation': 'statistical_32',
            'candidates': [
                {'C': C, 'gamma': gamma, 'class_weight': 'balanced'}
                for C, gamma in product([0.1, 1.0, 10.0], ['scale', 0.01])
            ],
        },
        'stat_random_forest': {
            'representation': 'statistical_32',
            'candidates': [
                {
                    'n_estimators': 300,
                    'max_depth': max_depth,
                    'min_samples_leaf': min_samples_leaf,
                    'class_weight': 'balanced',
                    'n_jobs': 1,
                }
                for max_depth, min_samples_leaf in product([None, 6, 12], [1, 3])
            ],
        },
        'stat_hist_gradient_boosting': {
            'representation': 'statistical_32',
            'candidates': [
                {
                    'learning_rate': learning_rate,
                    'max_leaf_nodes': max_leaf_nodes,
                    'l2_regularization': l2_regularization,
                    'max_iter': 200,
                }
                for learning_rate, max_leaf_nodes, l2_regularization in product([0.05, 0.1], [7, 15], [0.0, 1.0])
            ],
        },
        'flat_logreg': {
            'representation': 'flattened_120',
            'candidates': [
                {'C': C, 'class_weight': class_weight, 'solver': 'lbfgs', 'max_iter': 5000}
                for C, class_weight in product([0.1, 1.0, 10.0], [None, 'balanced'])
            ],
        },
    }

def baseline_candidate_id(family_key: str, params: dict) -> str:
    if family_key in {'stat_logreg', 'flat_logreg'}:
        cw = 'none' if params['class_weight'] is None else 'balanced'
        return f"{family_key.upper()}__C{params['C']}__CW{cw}"
    if family_key == 'stat_svm':
        gamma = params['gamma'] if params['gamma'] == 'scale' else f"{params['gamma']:.2f}"
        return f"STAT_SVM__C{params['C']}__G{gamma}"
    if family_key == 'stat_random_forest':
        depth = 'none' if params['max_depth'] is None else str(params['max_depth'])
        return f"STAT_RF__D{depth}__L{params['min_samples_leaf']}"
    if family_key == 'stat_hist_gradient_boosting':
        return f"STAT_HGB__LR{params['learning_rate']}__LN{params['max_leaf_nodes']}__L2{params['l2_regularization']}"
    raise ValueError(f'Unknown family_key: {family_key}')

def make_comparison_manifest() -> dict:
    manifest = {
        'protocol_version': 'publication_protocol_v2',
        'schema_version': 1,
        'seed': SEED,
        'k_folds': K_FOLDS,
        'grouping_protocol': 'GroupKFold grouped by training subject IDs only',
        'baseline_families': {
            family_key: {
                'display_name': family_display_name(family_key),
                'representation': spec['representation'],
                'candidate_grid': spec['candidates'],
                'candidate_count': len(spec['candidates']),
            }
            for family_key, spec in baseline_grid().items()
        },
        'feature_representations': {
            'statistical_32': {
                'shape': 32,
                'signals': ['temp', 'hr', 'gsr', 'rr'],
                'features_per_signal': ['mean', 'std', 'median', 'min', 'max', 'iqr', 'slope', 'mean_abs_diff'],
            },
            'flattened_120': {
                'shape': 120,
                'signals': ['temp', 'hr', 'gsr', 'rr'],
                'time_steps': 30,
            },
        },
        'primary_selection_metric': PRIMARY_SELECTION_METRIC,
        'tie_break_rules': TIE_BREAK_RULES,
        'final_reporting_metrics': FINAL_REPORTING_METRICS,
        'predefined_ablations': ['ABLATION_FULL', 'ABLATION_NO_BN', 'ABLATION_NO_ANFIS', 'ABLATION_NO_FTS'],
        'planned_statistical_analyses': {
            'clustered_bootstrap_confidence_intervals': {
                'level': 0.95,
                'resampling_unit': 'subject',
                'replicates': BOOTSTRAP_REPLICATES,
                'seed': BOOTSTRAP_SEED,
                'undefined_auc_handling': 'skip replicate and report effective replicate count',
            },
            'paired_significance_test': 'McNemar test between the proposed model and the best locked classical baseline',
        },
        'heldout_results_used': False,
        'heldout_predictions_used': False,
        'manifest_freeze_statement': 'Final comparison manifest frozen before baseline execution.',
    }
    manifest['manifest_hash'] = stable_hash({k: v for k, v in manifest.items() if k != 'manifest_hash'})
    return manifest

def verify_json_hashes(payload: dict, *, hash_key: str, ignore_keys: tuple[str, ...] = ()) -> bool:
    expected = stable_hash({k: v for k, v in payload.items() if k != hash_key and k not in ignore_keys})
    return payload[hash_key] == expected

print('PROTOCOL MODE = BASELINE_DEVELOPMENT_ONLY')
print('FINAL TEST ACCESS = DISABLED')
print(f'project_root = {PROJECT_ROOT}')
print(f'seed = {SEED}')
print(f'fold_count = {K_FOLDS}')
print(f'comparison_dir = {COMPARISON_DIR}')


PROTOCOL MODE = BASELINE_DEVELOPMENT_ONLY
FINAL TEST ACCESS = DISABLED
project_root = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final
seed = 42
fold_count = 5
comparison_dir = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final\artifacts\publication_protocol_v2\comparison


In [2]:
proposed_lock = read_json(PROPOSED_LOCK_PATH)
development_manifest = read_json(DEVELOPMENT_DIR / 'publication_candidate_manifest.json')
development_summary = read_json(DEVELOPMENT_DIR / 'publication_development_summary.json')
runner_source = (PROJECT_ROOT / 'src' / 'fbn_pipeline' / 'runner.py').read_text(encoding='utf-8')
proposed_required_fields = {
    'protocol_version', 'schema_version', 'candidate_manifest_hash', 'configuration_id', 'config', 'config_hash',
    'lock_hash', 'oof_metrics', 'threshold', 'threshold_source', 'seed', 'k_folds', 'grouping_protocol',
    'selection_metric', 'tie_break_rules', 'selected_using', 'final_test_evaluated'
}
missing_proposed_fields = sorted(proposed_required_fields.difference(proposed_lock))
assert not missing_proposed_fields, missing_proposed_fields
assert proposed_lock['selected_using'] == 'training_oof_only'
assert proposed_lock['threshold_source'] == 'training_oof_only'
assert proposed_lock['final_test_evaluated'] is False
assert proposed_lock['candidate_manifest_hash'] == development_manifest['manifest_hash']
assert proposed_lock['config_hash'] == config_hash_from_dict(proposed_lock['config'])
assert verify_json_hashes(proposed_lock, hash_key='lock_hash')
assert 'ExperimentConfig(**lock_payload["config"])' in runner_source
assert 'apply_preset' not in runner_source.split('def run_final_evaluation_from_lock', 1)[1].split('def write_development_results_csv', 1)[0]
fts_df = pd.read_csv(DEVELOPMENT_DIR / 'fts_selection_oof.csv')
rule_df = pd.read_csv(DEVELOPMENT_DIR / 'anfis_rule_selection_oof.csv')
bn_df = pd.read_csv(DEVELOPMENT_DIR / 'bn_selection_oof.csv')
assert set(fts_df['configuration_id']) == {'EXP1_MF_TRI', 'EXP1_MF_GAUSS', 'EXP1_MF_TRAP', 'EXP1_MF_GBELL'}
assert set(rule_df['rule_count']) == {3, 6, 8, 12}
bn_param_strings = set(bn_df['bn_parameters'])
assert len(bn_param_strings) == len(bn_df)
assert development_manifest['frozen_before_execution'] is True
assert development_manifest['candidate_counts']['stage_a_executed'] == 4
assert development_manifest['candidate_counts']['stage_b_branch_size'] == 4
assert development_manifest['candidate_counts']['stage_c_branch_size'] == 4
assert 'stage_b_candidates_by_stage_a' in development_manifest and 'stage_c_candidates_by_stage_a_and_stage_b' in development_manifest
print(json.dumps({
    'proposed_lock_verified': True,
    'stage_a_distinct_mf_count': int(fts_df['configuration_id'].nunique()),
    'stage_b_distinct_rule_count': int(rule_df['rule_count'].nunique()),
    'stage_c_distinct_bn_count': int(bn_df['configuration_id'].nunique()),
    'proposed_lock_hash': proposed_lock['lock_hash'],
    'proposed_config_hash': proposed_lock['config_hash'],
    'proposed_selected_using': proposed_lock['selected_using'],
    'proposed_threshold_source': proposed_lock['threshold_source'],
    'proposed_final_test_evaluated': proposed_lock['final_test_evaluated'],
}, indent=2))


{
  "proposed_lock_verified": true,
  "stage_a_distinct_mf_count": 4,
  "stage_b_distinct_rule_count": 4,
  "stage_c_distinct_bn_count": 4,
  "proposed_lock_hash": "a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37",
  "proposed_config_hash": "ad2f91375057dbe40890c902ff39bdaf8c32a3b31a588f745e2aed5eba82706e",
  "proposed_selected_using": "training_oof_only",
  "proposed_threshold_source": "training_oof_only",
  "proposed_final_test_evaluated": false
}


In [3]:
training_split = load_training_split(PROJECT_ROOT / 'last_30s_segments')
X_train = training_split.X_train
y_train = training_split.y_train
user_train = training_split.user_train
label_counts = {
    'rest': int(np.sum(y_train == 0)),
    'task': int(np.sum(y_train == 1)),
}
train_subject_ids = sorted(np.unique(user_train).tolist())
print(f'training_shape = {X_train.shape}')
print(f'training_label_counts = {label_counts}')
print(f'training_subject_ids = {train_subject_ids}')


training_shape = (632, 4, 30)
training_label_counts = {'rest': 317, 'task': 315}
training_subject_ids = ['1mpau', '2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']


In [4]:
verification_stage_ids = {
    'stage_a': fts_df['configuration_id'].tolist(),
    'stage_b': rule_df['configuration_id'].tolist(),
    'stage_c': bn_df['configuration_id'].tolist(),
}
verification_results = {}
for stage_name, candidate_ids in verification_stage_ids.items():
    stage_results = []
    for candidate_id in candidate_ids:
        if stage_name == 'stage_a':
            cfg_dict = development_manifest['stage_a_candidates'][candidate_id]
        elif stage_name == 'stage_b':
            selected_stage_a_id = development_summary['selected_fts_candidate']
            cfg_dict = development_manifest['stage_b_candidates_by_stage_a'][selected_stage_a_id][candidate_id]
        else:
            selected_stage_a_id = development_summary['selected_fts_candidate']
            selected_stage_b_id = development_summary['selected_anfis_candidate']
            cfg_dict = development_manifest['stage_c_candidates_by_stage_a_and_stage_b'][selected_stage_a_id][selected_stage_b_id][candidate_id]
        cfg = ExperimentConfig(**cfg_dict)
        result = run_development(
            X_train,
            y_train,
            user_train,
            cfg,
            configuration_id=candidate_id,
            print_progress=False,
        )
        stage_results.append({
            'configuration_id': candidate_id,
            'config_hash': stable_hash(cfg.to_dict()),
            'oof_hash': array_hash(result.oof_post),
            'threshold': result.threshold,
        })
    verification_results[stage_name] = pd.DataFrame(stage_results)
    assert verification_results[stage_name]['config_hash'].nunique() == len(stage_results)
    assert verification_results[stage_name]['oof_hash'].nunique() > 1
print('candidate_diversity_verification')
for stage_name, df in verification_results.items():
    print(stage_name)
    print(df.to_string(index=False))


--------------------------------------------------------------------------------
  Fold | Split | Total | cls 0 | cls 1 | Users
--------------------------------------------------------------------------------
     1 | Train |   504 |   251 |   253 | ['1mpau', '2nxs5', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']
       |   Val |   128 |    66 |    62 | ['5gpsc', 'hpbxa', 'ibvx8', 'iz2ps']
--------------------------------------------------------------------------------
  Fold  1 | Train | Acc: 0.7004 | AUC: 0.7761 | F1: 0.7289
         |  Val  | Acc: 0.5469 | AUC: 0.5367 | F1: 0.5735
--------------------------------------------------------------------------------
     2 | Train |   517 |   260 |   257 | ['2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl']
       |   Val |   115 |    57 |    58 | ['1mpau', 'wjxci', 'yljm5']
--------------------

In [5]:
comparison_manifest = make_comparison_manifest()
COMPARISON_MANIFEST_PATH.write_text(json.dumps(comparison_manifest, indent=2, sort_keys=True), encoding='utf-8')
comparison_manifest = read_json(COMPARISON_MANIFEST_PATH)
assert verify_json_hashes(comparison_manifest, hash_key='manifest_hash')
print(json.dumps({
    'comparison_manifest_path': str(COMPARISON_MANIFEST_PATH),
    'comparison_manifest_hash': comparison_manifest['manifest_hash'],
    'baseline_family_counts': {k: v['candidate_count'] for k, v in comparison_manifest['baseline_families'].items()},
    'heldout_results_used': comparison_manifest['heldout_results_used'],
}, indent=2))


{
  "comparison_manifest_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\comparison\\final_comparison_manifest.json",
  "comparison_manifest_hash": "7538e7e8521030e35d02e56684d11b40afd456795ab9d62c15936269eab68a9f",
  "baseline_family_counts": {
    "flat_logreg": 6,
    "stat_hist_gradient_boosting": 8,
    "stat_logreg": 6,
    "stat_random_forest": 6,
    "stat_svm": 6
  },
  "heldout_results_used": false
}


In [6]:
stat_features, stat_feature_spec = build_feature_matrix(X_train, 'statistical_32')
flat_features, flat_feature_spec = build_feature_matrix(X_train, 'flattened_120')
assert stat_features.shape == (len(X_train), 32)
assert flat_features.shape == (len(X_train), 120)
print(json.dumps({
    'statistical_feature_shape': list(stat_features.shape),
    'flattened_feature_shape': list(flat_features.shape),
    'stat_feature_spec': stat_feature_spec,
    'flat_feature_spec': flat_feature_spec,
}, indent=2))


{
  "statistical_feature_shape": [
    632,
    32
  ],
  "flattened_feature_shape": [
    632,
    120
  ],
  "stat_feature_spec": {
    "representation": "statistical_32",
    "shape": 32,
    "per_signal_features": [
      "mean",
      "std",
      "median",
      "min",
      "max",
      "iqr",
      "slope",
      "mean_abs_diff"
    ],
    "signals": [
      "temp",
      "hr",
      "gsr",
      "rr"
    ]
  },
  "flat_feature_spec": {
    "representation": "flattened_120",
    "shape": 120,
    "signals": [
      "temp",
      "hr",
      "gsr",
      "rr"
    ],
    "time_steps": 30
  }
}


In [7]:
baseline_specs = baseline_grid()
all_baseline_results = []
family_winner_results = []
baseline_lock_payloads = {}
for family_key, spec in baseline_specs.items():
    family_results = []
    for params in spec['candidates']:
        candidate_id = baseline_candidate_id(family_key, params)
        result = run_baseline_development(
            X_train,
            y_train,
            user_train,
            family=family_key,
            representation=spec['representation'],
            hyperparameters=params,
            configuration_id=candidate_id,
            seed=SEED,
            k_folds=K_FOLDS,
        )
        family_results.append(result)
        all_baseline_results.append(result)
    selection = select_best_baseline(family_results)
    winner = selection.selected_result
    family_winner_results.append(winner)
    baseline_lock_path = BASELINE_LOCK_DIR / f'{family_key}.json'
    payload = write_baseline_lock(
        winner,
        comparison_manifest_hash=comparison_manifest['manifest_hash'],
        output_path=baseline_lock_path,
    )
    baseline_lock_payloads[family_key] = payload
baseline_candidate_rows = []
for result in all_baseline_results:
    row = result.as_row()
    row['family_display_name'] = family_display_name(result.family)
    row['selected'] = any(result.configuration_id == winner.configuration_id for winner in family_winner_results)
    baseline_candidate_rows.append(row)
baseline_candidates_df = pd.DataFrame(baseline_candidate_rows)
baseline_candidates_df['family_rank'] = (
    baseline_candidates_df
    .sort_values(['baseline_family', 'oof_balanced_accuracy', 'oof_macro_f1', 'oof_auc', 'configuration_id'], ascending=[True, False, False, False, True])
    .groupby('baseline_family')
    .cumcount() + 1
)
baseline_candidates_df = baseline_candidates_df.sort_values(['baseline_family', 'family_rank']).reset_index(drop=True)
assert not any(col.startswith('test_') for col in baseline_candidates_df.columns)
BASELINE_CANDIDATES_CSV_PATH.write_text(baseline_candidates_df.to_csv(index=False), encoding='utf-8')
baseline_winners_df = pd.DataFrame([
    {
        'baseline_family': result.family,
        'family_display_name': family_display_name(result.family),
        'configuration_id': result.configuration_id,
        'selected_parameters': json.dumps(result.hyperparameters, sort_keys=True),
        'input_representation': result.representation,
        'oof_accuracy': result.oof_metrics['accuracy'],
        'oof_balanced_accuracy': result.oof_metrics['bal_acc'],
        'oof_precision': result.oof_metrics['precision'],
        'oof_recall': result.oof_metrics['recall'],
        'oof_task_f1': result.oof_metrics['f1'],
        'oof_macro_f1': result.oof_metrics['macro_f1'],
        'oof_auc': result.oof_metrics['auc'],
        'selected_threshold': result.threshold,
        'prediction_hash': result.prediction_hash,
        'config_hash': result.config_hash,
    }
    for result in family_winner_results
]).sort_values(['oof_balanced_accuracy', 'oof_macro_f1', 'oof_auc', 'configuration_id'], ascending=[False, False, False, True]).reset_index(drop=True)
assert not any(col.startswith('test_') for col in baseline_winners_df.columns)
BASELINE_WINNERS_CSV_PATH.write_text(baseline_winners_df.to_csv(index=False), encoding='utf-8')
print('Baseline candidate rankings (OOF only)')
print(baseline_candidates_df[['baseline_family', 'configuration_id', 'family_rank', 'oof_balanced_accuracy', 'oof_macro_f1', 'oof_auc', 'selected_threshold', 'selected']].to_string(index=False))
print('Baseline family winners (OOF only)')
print(baseline_winners_df.to_string(index=False))


Baseline candidate rankings (OOF only)
            baseline_family               configuration_id  family_rank  oof_balanced_accuracy  oof_macro_f1  oof_auc  selected_threshold  selected
                flat_logreg      FLAT_LOGREG__C0.1__CWnone            1                 0.5722        0.5240   0.5728            0.378629      True
                flat_logreg  FLAT_LOGREG__C0.1__CWbalanced            2                 0.5706        0.5228   0.5731            0.383934     False
                flat_logreg  FLAT_LOGREG__C1.0__CWbalanced            3                 0.5674        0.5273   0.5627            0.367837     False
                flat_logreg      FLAT_LOGREG__C1.0__CWnone            4                 0.5658        0.5251   0.5624            0.367423     False
                flat_logreg FLAT_LOGREG__C10.0__CWbalanced            5                 0.5606        0.5488   0.5517            0.451818     False
                flat_logreg     FLAT_LOGREG__C10.0__CWnone            6  

In [8]:
best_classical_baseline_selection = select_best_baseline(family_winner_results)
best_classical_baseline = best_classical_baseline_selection.selected_result
print(json.dumps({
    'best_classical_baseline_family': best_classical_baseline.family,
    'best_classical_baseline_id': best_classical_baseline.configuration_id,
    'best_classical_baseline_metrics': best_classical_baseline.oof_metrics,
    'best_classical_baseline_threshold': best_classical_baseline.threshold,
}, indent=2))


{
  "best_classical_baseline_family": "stat_logreg",
  "best_classical_baseline_id": "STAT_LOGREG__C0.1__CWnone",
  "best_classical_baseline_metrics": {
    "accuracy": 0.6329,
    "precision": 0.6293,
    "recall": 0.6413,
    "f1": 0.6352,
    "auc": 0.647,
    "acc": 0.6329,
    "bal_acc": 0.6329,
    "macro_f1": 0.6329
  },
  "best_classical_baseline_threshold": 0.533262
}


In [9]:
proposed_cfg = ExperimentConfig(**proposed_lock['config'])
ablation_templates = development_manifest['stage_d_ablation_templates']
ablation_lock_payloads = {}
for ablation_id, overrides in ablation_templates.items():
    cfg = proposed_cfg.merged(
        **overrides,
        experiment_group='PUB_ABLATION_LOCK',
        variant_name=ablation_id,
    )
    result = run_development(
        X_train,
        y_train,
        user_train,
        cfg,
        configuration_id=ablation_id,
        print_progress=False,
    )
    selection = select_best_development_config([result])
    payload = lock_development_config(
        result,
        selection,
        output_path=ABLATION_LOCK_DIR / f'{ablation_id}.json',
        extra_metadata={
            'comparison_manifest_hash': comparison_manifest['manifest_hash'],
            'source_publication_manifest_hash': proposed_lock['candidate_manifest_hash'],
            'lock_role': 'predefined_ablation',
        },
    )
    ablation_lock_payloads[ablation_id] = payload
    assert payload['final_test_evaluated'] is False
print(json.dumps({
    'ablation_lock_paths': {k: str(ABLATION_LOCK_DIR / f'{k}.json') for k in ablation_lock_payloads},
    'ablation_lock_hashes': {k: v['lock_hash'] for k, v in ablation_lock_payloads.items()},
}, indent=2))


--------------------------------------------------------------------------------
  Fold | Split | Total | cls 0 | cls 1 | Users
--------------------------------------------------------------------------------
     1 | Train |   504 |   251 |   253 | ['1mpau', '2nxs5', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']
       |   Val |   128 |    66 |    62 | ['5gpsc', 'hpbxa', 'ibvx8', 'iz2ps']
--------------------------------------------------------------------------------
  Fold  1 | Train | Acc: 0.7004 | AUC: 0.7761 | F1: 0.7289
         |  Val  | Acc: 0.5469 | AUC: 0.5367 | F1: 0.5735
--------------------------------------------------------------------------------
     2 | Train |   517 |   260 |   257 | ['2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl']
       |   Val |   115 |    57 |    58 | ['1mpau', 'wjxci', 'yljm5']
--------------------

In [10]:
statistical_analysis_plan = {
    'protocol_version': 'publication_protocol_v2',
    'schema_version': 1,
    'comparison_manifest_hash': comparison_manifest['manifest_hash'],
    'proposed_model_lock_path': str(PROPOSED_LOCK_PATH),
    'best_classical_baseline_lock_path': str(BASELINE_LOCK_DIR / f'{best_classical_baseline.family}.json'),
    'predefined_ablation_lock_paths': {k: str(ABLATION_LOCK_DIR / f'{k}.json') for k in ablation_lock_payloads},
    'final_reporting_metrics': FINAL_REPORTING_METRICS,
    'clustered_bootstrap_confidence_intervals': {
        'level': 0.95,
        'resampling_unit': 'subject',
        'replicates': BOOTSTRAP_REPLICATES,
        'seed': BOOTSTRAP_SEED,
        'sampling_scheme': 'sample training-independent subjects with replacement and aggregate window-level predictions within each replicate',
        'undefined_auc_handling': 'skip undefined AUC replicates and report effective replicate count',
    },
    'paired_significance_test': {
        'name': 'McNemar',
        'comparison': 'proposed model versus best locked classical baseline',
        'execution_status': 'not_executed',
    },
}
statistical_analysis_plan['plan_hash'] = stable_hash({k: v for k, v in statistical_analysis_plan.items() if k != 'plan_hash'})
STAT_PLAN_JSON_PATH.write_text(json.dumps(statistical_analysis_plan, indent=2, sort_keys=True), encoding='utf-8')
STAT_PLAN_MD_PATH.write_text(
    '\n'.join([
        '# Statistical Analysis Plan',
        '',
        f"- Comparison manifest hash: {comparison_manifest['manifest_hash']}",
        f"- Proposed model lock: {PROPOSED_LOCK_PATH}",
        f"- Best classical baseline lock: {BASELINE_LOCK_DIR / (best_classical_baseline.family + '.json')}",
        f"- Bootstrap confidence intervals: 95% subject-clustered, {BOOTSTRAP_REPLICATES} replicates, seed {BOOTSTRAP_SEED}",
        '- Undefined AUC handling: skip undefined replicates and report the effective replicate count.',
        '- Paired significance test: McNemar test between the proposed model and the best locked classical baseline.',
        '- Execution status: frozen only, not executed.',
    ]),
    encoding='utf-8',
)
print(json.dumps(statistical_analysis_plan, indent=2))


{
  "protocol_version": "publication_protocol_v2",
  "schema_version": 1,
  "comparison_manifest_hash": "7538e7e8521030e35d02e56684d11b40afd456795ab9d62c15936269eab68a9f",
  "proposed_model_lock_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\locked_config.json",
  "best_classical_baseline_lock_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\baselines\\locks\\stat_logreg.json",
  "predefined_ablation_lock_paths": {
    "ABLATION_FULL": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\ablations\\locks\\ABLATION_FULL.json",
    "ABLATION_NO_ANFIS": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\ablations\\locks\\ABLATION_NO_ANFIS.json",
    "ABLATION_NO_BN": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognit

In [11]:
baseline_summary = {
    'comparison_manifest_path': str(COMPARISON_MANIFEST_PATH),
    'comparison_manifest_hash': comparison_manifest['manifest_hash'],
    'baseline_family_winners': {
        result.family: {
            'configuration_id': result.configuration_id,
            'lock_path': str(BASELINE_LOCK_DIR / f'{result.family}.json'),
            'lock_hash': baseline_lock_payloads[result.family]['lock_hash'],
            'oof_metrics': result.oof_metrics,
            'threshold': result.threshold,
        }
        for result in family_winner_results
    },
    'best_classical_baseline': {
        'family': best_classical_baseline.family,
        'configuration_id': best_classical_baseline.configuration_id,
        'lock_path': str(BASELINE_LOCK_DIR / f'{best_classical_baseline.family}.json'),
        'lock_hash': baseline_lock_payloads[best_classical_baseline.family]['lock_hash'],
    },
    'ablation_locks': {
        key: {
            'lock_path': str(ABLATION_LOCK_DIR / f'{key}.json'),
            'lock_hash': value['lock_hash'],
        }
        for key, value in ablation_lock_payloads.items()
    },
    'heldout_evaluation_executed': False,
}
BASELINE_SUMMARY_JSON_PATH.write_text(json.dumps(baseline_summary, indent=2, sort_keys=True), encoding='utf-8')
BASELINE_SUMMARY_MD_PATH.write_text(
    '\n'.join([
        '# Baseline Development Summary',
        '',
        f"- Comparison manifest: {COMPARISON_MANIFEST_PATH}",
        f"- Comparison manifest hash: {comparison_manifest['manifest_hash']}",
        f"- Best classical baseline: {family_display_name(best_classical_baseline.family)} ({best_classical_baseline.configuration_id})",
        f"- Best classical baseline threshold: {best_classical_baseline.threshold}",
        '',
        'All proposed-model verification, controlled baseline selection, predefined ablation locking, and statistical-analysis planning were performed using the 18-subject development set only. The five held-out subjects were not evaluated during this phase.',
    ]),
    encoding='utf-8',
)
print(json.dumps(baseline_summary, indent=2))


{
  "comparison_manifest_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\comparison\\final_comparison_manifest.json",
  "comparison_manifest_hash": "7538e7e8521030e35d02e56684d11b40afd456795ab9d62c15936269eab68a9f",
  "baseline_family_winners": {
    "stat_logreg": {
      "configuration_id": "STAT_LOGREG__C0.1__CWnone",
      "lock_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\baselines\\locks\\stat_logreg.json",
      "lock_hash": "2797cf9d4683cec2b05e8328154a7a360ffe22526bff1c2772b5d53a945f949e",
      "oof_metrics": {
        "accuracy": 0.6329,
        "precision": 0.6293,
        "recall": 0.6413,
        "f1": 0.6352,
        "auc": 0.647,
        "acc": 0.6329,
        "bal_acc": 0.6329,
        "macro_f1": 0.6329
      },
      "threshold": 0.533262
    },
    "stat_svm": {
      "configuration_id": "STAT_SVM__C0.1__G0.01",
   

In [12]:
final_evaluation_called = False
baseline_candidate_columns_ok = not any(col.startswith('test_') for col in baseline_candidates_df.columns)
baseline_winner_columns_ok = not any(col.startswith('test_') for col in baseline_winners_df.columns)
heldout_artifact_path = ARTIFACT_ROOT / 'final_test_results.csv'
heldout_artifact_created = heldout_artifact_path.exists()
all_baseline_locks_ok = all(payload['final_test_evaluated'] is False for payload in baseline_lock_payloads.values())
all_ablation_locks_ok = all(payload['final_test_evaluated'] is False for payload in ablation_lock_payloads.values())
safeguards = {
    'no_test_columns_in_baseline_candidates': baseline_candidate_columns_ok,
    'no_test_columns_in_baseline_winners': baseline_winner_columns_ok,
    'train_only_loader_used': True,
    'final_evaluation_not_called': final_evaluation_called is False,
    'heldout_artifact_created': heldout_artifact_created,
    'all_baseline_locks_final_test_false': all_baseline_locks_ok,
    'all_ablation_locks_final_test_false': all_ablation_locks_ok,
}
assert all(value is True or value is False for value in safeguards.values())
assert safeguards['no_test_columns_in_baseline_candidates'] is True
assert safeguards['no_test_columns_in_baseline_winners'] is True
assert safeguards['train_only_loader_used'] is True
assert safeguards['final_evaluation_not_called'] is True
assert safeguards['heldout_artifact_created'] is False
assert safeguards['all_baseline_locks_final_test_false'] is True
assert safeguards['all_ablation_locks_final_test_false'] is True
print(json.dumps(safeguards, indent=2))


{
  "no_test_columns_in_baseline_candidates": true,
  "no_test_columns_in_baseline_winners": true,
  "train_only_loader_used": true,
  "final_evaluation_not_called": true,
  "heldout_artifact_created": false,
  "all_baseline_locks_final_test_false": true,
  "all_ablation_locks_final_test_false": true
}
